In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ── Sources ──────────────────────────────────────────────────────────
silver = spark.table("iran_israel_capstone_project.silver.daily_market_clean")
event_dim = spark.table("iran_israel_capstone_project.silver.event_dim")

# ════════════════════════════════════════════════════════════════════
# TABLE 1: gold_vix_daily  (KPI 1 — VIX Shock Detection)
# ════════════════════════════════════════════════════════════════════

# Filter to rows with VIX data, add row number for proximity joins
w_date = Window.orderBy("trade_date")

vix_daily = (
    silver
    .filter(F.col("indiavix_close").isNotNull())
    .select(
        "trade_date", "indiavix_close", "indiavix_20d_ma",
        "nifty_close", "nifty_daily_return_pct",
        "event_id", "event_type", "severity"
    )
    .withColumn("row_num", F.row_number().over(w_date))
)

# VIX-to-MA ratio and shock flag
vix_daily = vix_daily.withColumn(
    "vix_to_ma_ratio",
    F.round(F.col("indiavix_close") / F.col("indiavix_20d_ma"), 4)
).withColumn(
    "is_vix_shock",
    F.col("indiavix_close") >= 1.5 * F.col("indiavix_20d_ma")
).withColumn(
    "year_month",
    F.date_format("trade_date", "yyyy-MM")
)

# Write gold_vix_daily
vix_daily_out = vix_daily.select(
    "trade_date", "indiavix_close", "indiavix_20d_ma", "vix_to_ma_ratio",
    "is_vix_shock", "year_month",
    "nifty_close", "nifty_daily_return_pct",
    "event_id", "event_type", "severity"
)
vix_daily_out.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "iran_israel_capstone_project.gold.gold_vix_daily"
)
print(f"✅ gold_vix_daily written: {vix_daily_out.count()} rows")

# KPI 1 Summary: shock days per month
shock_summary = (
    vix_daily
    .groupBy("year_month")
    .agg(
        F.count("*").alias("trading_days"),
        F.sum(F.when(F.col("is_vix_shock"), 1).otherwise(0)).alias("shock_days"),
        F.round(F.max("vix_to_ma_ratio"), 2).alias("peak_vix_ratio")
    )
    .orderBy("year_month")
)
print("\n── KPI 1: VIX Shock Days per Month ──")
total_shocks = vix_daily.filter(F.col("is_vix_shock")).count()
print(f"   Total shock days (VIX >= 1.5× 20d MA): {total_shocks}")
display(shock_summary)

# ════════════════════════════════════════════════════════════════════
# TABLE 2: gold_vix_event_decay  (KPI 2 & 3)
# ════════════════════════════════════════════════════════════════════

# Get HIGH/CRITICAL events with their trade dates (row numbers)
hc_events = (
    event_dim
    .filter(F.col("severity").isin("HIGH", "CRITICAL"))
    .select("event_date", "event_id", "severity", "event_type")
    .distinct()
)

# Join events to vix_daily to get event-day row numbers and VIX levels
event_vix = (
    hc_events
    .join(
        vix_daily.select(
            F.col("trade_date").alias("evt_trade_date"),
            F.col("indiavix_close").alias("event_day_vix"),
            F.col("nifty_close").alias("event_day_nifty"),
            F.col("row_num").alias("event_row_num")
        ),
        F.col("event_date") == F.col("evt_trade_date"),
        "inner"
    )
)

# Get pre-event VIX: the trading day immediately before the event
pre_event = (
    vix_daily.select(
        F.col("row_num").alias("pre_row_num"),
        F.col("indiavix_close").alias("pre_event_vix"),
        F.col("nifty_close").alias("pre_event_nifty")
    )
)

event_vix = event_vix.join(
    pre_event,
    F.col("event_row_num") - 1 == F.col("pre_row_num"),
    "left"
)

# ── KPI 2: VIX Decay ─────────────────────────────────────────────
# For each event, find first day AFTER the event where VIX <= pre_event_vix * 1.10
post_event_days = (
    event_vix
    .crossJoin(
        vix_daily.select(
            F.col("trade_date").alias("post_date"),
            F.col("indiavix_close").alias("post_vix"),
            F.col("row_num").alias("post_row_num")
        )
    )
    .filter(
        (F.col("post_row_num") > F.col("event_row_num")) &
        (F.col("post_row_num") <= F.col("event_row_num") + 60)  # look up to 60 trading days
    )
    .filter(F.col("post_vix") <= F.col("pre_event_vix") * 1.10)
)

# First recovery day per event
w_event = Window.partitionBy("event_id").orderBy("post_row_num")
vix_recovery = (
    post_event_days
    .withColumn("rn", F.row_number().over(w_event))
    .filter(F.col("rn") == 1)
    .select(
        "event_id",
        F.col("post_date").alias("vix_recovery_date"),
        (F.col("post_row_num") - F.col("event_row_num")).alias("vix_decay_days")
    )
)

# ── KPI 3: Nifty Shock-Recovery ──────────────────────────────────
# For each event, find: (a) trough Nifty within 30 TDs, (b) first recovery day
post_nifty = (
    event_vix
    .crossJoin(
        vix_daily.select(
            F.col("trade_date").alias("nifty_post_date"),
            F.col("nifty_close").alias("nifty_post_close"),
            F.col("row_num").alias("nifty_post_row")
        )
    )
    .filter(
        (F.col("nifty_post_row") > F.col("event_row_num")) &
        (F.col("nifty_post_row") <= F.col("event_row_num") + 30)
    )
)

# Trough per event
nifty_trough = (
    post_nifty
    .groupBy("event_id", "pre_event_nifty")
    .agg(
        F.min("nifty_post_close").alias("nifty_trough"),
        F.min(F.when(F.col("nifty_post_close") == F.col("nifty_post_close"), F.col("nifty_post_date"))).alias("trough_date_approx")
    )
    .withColumn(
        "max_drawdown_pct",
        F.round((F.col("nifty_trough") / F.col("pre_event_nifty") - 1) * 100, 2)
    )
)

# First Nifty recovery day (close >= pre_event_nifty)
nifty_recov = (
    post_nifty
    .filter(F.col("nifty_post_close") >= F.col("pre_event_nifty"))
    .withColumn("rn", F.row_number().over(
        Window.partitionBy("event_id").orderBy("nifty_post_row")
    ))
    .filter(F.col("rn") == 1)
    .select(
        "event_id",
        F.col("nifty_post_date").alias("nifty_recovery_date"),
        (F.col("nifty_post_row") - F.col("event_row_num")).alias("nifty_recovery_days")
    )
)

# ── Assemble final event-level table ─────────────────────────────
event_decay = (
    event_vix
    .select(
        "event_id", "event_date", "severity", "event_type",
        "pre_event_vix", "event_day_vix",
        "pre_event_nifty", "event_day_nifty"
    )
    .join(vix_recovery, on="event_id", how="left")
    .join(nifty_trough.select("event_id", "nifty_trough", "max_drawdown_pct"), on="event_id", how="left")
    .join(nifty_recov, on="event_id", how="left")
    .withColumn(
        "vix_spike_pct",
        F.round((F.col("event_day_vix") / F.col("pre_event_vix") - 1) * 100, 2)
    )
    .withColumn(
        "nifty_recovered_within_30td",
        F.col("nifty_recovery_date").isNotNull()
    )
)

event_decay.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    "iran_israel_capstone_project.gold.gold_vix_event_decay"
)
print(f"\n✅ gold_vix_event_decay written: {event_decay.count()} rows")

# ── KPI 2 Summary: Average VIX Decay Period ──
recovered = event_decay.filter(F.col("vix_decay_days").isNotNull())
avg_decay = recovered.agg(F.round(F.avg("vix_decay_days"), 1).alias("avg")).collect()[0]["avg"]
median_decay = recovered.agg(F.expr("percentile_approx(vix_decay_days, 0.5)").alias("med")).collect()[0]["med"]
recovered_count = recovered.count()
total_events = event_decay.count()

print(f"\n── KPI 2: Post-Event VIX Decay ──")
print(f"   Events with VIX recovery: {recovered_count}/{total_events}")
print(f"   Avg decay period: {avg_decay} trading days")
print(f"   Median decay period: {median_decay} trading days")

# ── KPI 3 Summary: Shock-Recovery Pattern ──
recov_events = event_decay.filter(F.col("nifty_recovered_within_30td"))
recov_rate = round(recov_events.count() / total_events * 100, 1) if total_events > 0 else 0
avg_recov_days = recov_events.agg(F.round(F.avg("nifty_recovery_days"), 1).alias("avg")).collect()[0]["avg"]

print(f"\n── KPI 3: Nifty Shock-Recovery ──")
print(f"   Events where Nifty recovered within 30 TDs: {recov_events.count()}/{total_events} = {recov_rate}%")
print(f"   Avg recovery period: {avg_recov_days} trading days")

# April 2024 specific check
apr_event = event_decay.filter(F.col("event_id") == "IRN-ISR-002")
if apr_event.count() > 0:
    row = apr_event.collect()[0]
    print(f"\n   April 2024 (IRN-ISR-002):")
    print(f"     Pre-event Nifty: {row['pre_event_nifty']:.0f}")
    print(f"     Trough Nifty:    {row['nifty_trough']:.0f} (drawdown: {row['max_drawdown_pct']}%)")
    print(f"     Recovered:       {'Yes' if row['nifty_recovered_within_30td'] else 'No'}")
    if row['nifty_recovery_days']:
        print(f"     Recovery took:   {row['nifty_recovery_days']} trading days")

print("\n── Full Event Decay Table ──")
display(event_decay.orderBy("event_date"))